In [1]:
def link_edges(pair_list):
    edge_set = set()
    for u, v in pair_list:
        edge_set.add((u, v))
        edge_set.add((v, u))
    return edge_set

def is_maximal(I, P, A, edges):
    active_pool = set(P) & set(A)
    remaining = active_pool - set(I)
    for v in remaining:
        if all((v, u) not in edges for u in I):
            return False
    return True

def calculate_candidate_set(P, I, E):
    if not I:
        return tuple(sorted(list(P)))
    max_i = max(I)
    candidates = [
        v for v in P if v not in I
        if v > max_i and all((u, v) not in E for u in I)
    ]
    return tuple(sorted(candidates))

def print_traced_table(title, history, chromatic_num, solution_count):
    print(f"\n{'='*105}\n  TRACE LOG: {title}\n{'='*105}")
    print(f"{'State':<8} | {'Horizon (A)':<14} | {'Configuration (P, R, I, C)':<56} | {'Transition':<11}")
    print("-" * 105)
    for idx, item in enumerate(history, 1):
        scope, p, r, i, c, trans = item
        p_str = f"P={set(p)}" if p else "P=\u2205"
        r_str = f"R={list(r)}"
        i_str = f"I={set(i)}" if i else "I=\u2205"
        c_str = f"C={set(c)}" if c else "C=\u2205"
        config_str = f"({p_str}, {r_str}, {i_str}, {c_str})"
        print(f"Gamma_{idx:<2} | {str(list(scope)):<14} | {config_str:<56} | {trans:<11}")
    print("-" * 105)
    print(f" >> Calculated Chromatic Number (\u03c7) = {chromatic_num}")
    print(f" >> Total Distinct Terminal Graph Colorings = {solution_count}")
    print(f" >> Total Explored State Configurations = {len(history)}\n")

# -------------------------------------------------------------
# FIXED ATOMIC PURE DOWNWARD ENGINE
# -------------------------------------------------------------
def trace_pure_downward():
    V = [0, 1, 2, 3]
    E = link_edges([(0,1), (0,2), (1,2), (2,3)])

    history = []
    global_horizon = set(V)

    init_p = tuple(sorted(V))
    init_c = calculate_candidate_set(init_p, (), E)

    frontier = [(init_p, (), (), init_c, "Root")]
    min_colors = len(V) + 1

    while frontier:
        p, r, i, c, trans = frontier.pop(0)
        history.append((global_horizon, p, r, i, c, trans))

        # --- Multi-Layered Pruning Check ---
        # Change to strictly greater than (>) to let equal-sized alternate optimal paths complete
        if len(r) > min_colors:
            history[-1] = (global_horizon, p, r, i, c, "Pruned")
            continue

        # --- Terminal Complete Contraction Bounding Check ---
        if len(p) == 0:
            if len(r) < min_colors:
                min_colors = len(r)
            history[-1] = (global_horizon, p, r, i, c, "Terminal")
            continue

        # --- Transition Logic Flow ---
        if len(c) > 0:
            sorted_c = sorted(list(c))
            for v in sorted_c:
                next_i = tuple(sorted(list(i) + [v]))
                next_c = calculate_candidate_set(p, next_i, E)
                frontier.append((p, r, next_i, next_c, f"add({v})"))
        else:
            if len(i) > 0 and is_maximal(i, p, global_horizon, E):
                next_p = tuple(x for x in p if x not in i)
                next_c = calculate_candidate_set(next_p, (), E)
                frontier.append((next_p, r + (i,), (), next_c, "commit"))
            else:
                history[-1] = (global_horizon, p, r, i, c, "Dead End")

    terminals = [h for h in history if h[5] == "Terminal"]
    chi = min(len(t[2]) for t in terminals) if terminals else 0
    print_traced_table("PURE DOWNWARD STATIC ENGINE (ALL OPTIMAL SOLUTIONS)", history, chi, len(terminals))

if __name__ == "__main__":
    trace_pure_downward()


  TRACE LOG: PURE DOWNWARD STATIC ENGINE (ALL OPTIMAL SOLUTIONS)
State    | Horizon (A)    | Configuration (P, R, I, C)                               | Transition 
---------------------------------------------------------------------------------------------------------
Gamma_1  | [0, 1, 2, 3]   | (P={0, 1, 2, 3}, R=[], I=∅, C={0, 1, 2, 3})              | Root       
Gamma_2  | [0, 1, 2, 3]   | (P={0, 1, 2, 3}, R=[], I={0}, C={3})                     | add(0)     
Gamma_3  | [0, 1, 2, 3]   | (P={0, 1, 2, 3}, R=[], I={1}, C={3})                     | add(1)     
Gamma_4  | [0, 1, 2, 3]   | (P={0, 1, 2, 3}, R=[], I={2}, C=∅)                       | add(2)     
Gamma_5  | [0, 1, 2, 3]   | (P={0, 1, 2, 3}, R=[], I={3}, C=∅)                       | Dead End   
Gamma_6  | [0, 1, 2, 3]   | (P={0, 1, 2, 3}, R=[], I={0, 3}, C=∅)                    | add(3)     
Gamma_7  | [0, 1, 2, 3]   | (P={0, 1, 2, 3}, R=[], I={1, 3}, C=∅)                    | add(3)     
Gamma_8  | [0, 1, 2, 3]   | (P={0, 1

In [2]:
import os
import time
from google.colab import drive

# 1. Mount Google Drive to access the benchmark files
try:
    drive.mount('/content/drive')
except Exception as e:
    print("Proceeding assuming drive is already mounted or local environment is configured.")

# Define directory path where the DIMACS .col files are located
DIMACS_DIR = "/content/drive/MyDrive/GPU/DIMACS_all_ascii"

# Target ground-truth chromatic numbers for known DIMACS instances to evaluate outcome
DIMACS_TARGETS = {
    # Small Instances
    "myciel3.col": 4,
    "myciel4.col": 5,
    "queen5_5.col": 5,
    # Medium Instances
    "myciel5.col": 6,
    "queen6_6.col": 6,
    "queen7_7.col": 7,
    "le450_5a.col": 5,
    "le450_5b.col": 5,
    # Large Instances
    "myciel6.col": 7,
    "myciel7.col": 8,
    "queen8_8.col": 9,
    "queen9_9.col": 10,
    "le450_15a.col": 15,
    "le450_25a.col": 25,
}

def parse_dimacs_col(file_path):
    """Parses a standard DIMACS .col file and returns V and E."""
    vertices = set()
    edges = set()

    if not os.path.exists(file_path):
        return None, None

    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("c"):
                continue
            parts = line.split()
            if parts[0] == "p" and parts[1] == "edge":
                num_v = int(parts[2])
                # Ensure all vertices 0 to num_v-1 are present
                for i in range(num_v):
                    vertices.add(i)
            elif parts[0] == "e":
                # Convert standard 1-based DIMACS indexing to 0-based indexing
                u = int(parts[1]) - 1
                v = int(parts[2]) - 1
                vertices.add(u)
                vertices.add(v)
                edges.add((u, v))
                edges.add((v, u))

    return sorted(list(vertices)), edges

def is_maximal(I, P, edges):
    """Pure Downward relative maximality check over active pool P."""
    remaining = set(P) - set(I)
    for v in remaining:
        if all((v, u) not in edges for u in I):
            return False
    return True

def calculate_candidate_set(P, I, E):
    """Calculates canonical candidate sets using descending node constraints."""
    if not I:
        return tuple(sorted(list(P)))
    max_i = max(I)
    candidates = [
        v for v in P if v not in I
        if v > max_i and all((u, v) not in E for u in I)
    ]
    return tuple(sorted(candidates))

def run_pure_downward_engine(V, E, target_chi, timeout_seconds=60):
    """
    Executes the static pure downward transition system engine
    with precise time-tracking limits and branch assessments.
    """
    start_time = time.time()

    init_p = tuple(sorted(V))
    init_c = calculate_candidate_set(init_p, (), E)

    # Using LIFO queue (Stack) for efficient Deep Memory Traversal (DFS)
    frontier = [(init_p, (), (), init_c)]
    min_colors = len(V) + 1

    explored_states = 0
    space_fully_explored = False
    timeout_reached = False

    while frontier:
        # Check timeout boundary condition at every state transition loop
        if time.time() - start_time >= timeout_seconds:
            timeout_reached = True
            break

        p, r, i, c = frontier.pop()  # DFS extraction behavior
        explored_states += 1

        # --- Multi-Layered Pruning Check ---
        if len(r) > min_colors:
            continue

        # --- Terminal Complete Contraction Bounding Check ---
        if len(p) == 0:
            if len(r) < min_colors:
                min_colors = len(r)
                # Found exact target answer early, we can exit safely
                if min_colors == target_chi:
                    space_fully_explored = True # verified matching upper bound
            continue

        # --- Transition Logic Flow ---
        if len(c) > 0:
            for v in sorted(list(c), reverse=True):  # Reversed stack insertion preserves sorted processing
                next_i = tuple(sorted(list(i) + [v]))
                next_c = calculate_candidate_set(p, next_i, E)
                frontier.append((p, r, next_i, next_c))
        else:
            if len(i) > 0 and is_maximal(i, p, E):
                next_p = tuple(x for x in p if x not in i)
                next_c = calculate_candidate_set(next_p, (), E)
                frontier.append((next_p, r + (i,), (), next_c))

    if not frontier and not timeout_reached:
        space_fully_explored = True

    execution_time = time.time() - start_time
    calculated_chi = min_colors if min_colors <= len(V) else None

    # --- Structural Classification Branching Logic ---
    if calculated_chi == target_chi and not timeout_reached:
        status = "Success"
    elif calculated_chi == target_chi and timeout_reached:
        status = "Success with Timeout"
    elif calculated_chi != target_chi and timeout_reached:
        status = "Timeout"
    elif space_fully_explored and calculated_chi != target_chi:
        status = "Fail"
    else:
        status = "Timeout"

    return status, calculated_chi, explored_states, execution_time

def main():
    print(f"Scanning configuration benchmarks folder: {DIMACS_DIR}\n")
    if not os.path.exists(DIMACS_DIR):
        print(f"Error: Path '{DIMACS_DIR}' not found. Please verify your Google Drive directory.")
        return

    # Gather matching target files and split into size categories for ordered execution
    found_files = [f for f in os.listdir(DIMACS_DIR) if f.endswith(".col") and f in DIMACS_TARGETS]

    # Sorting protocol to guarantee small -> medium -> large transition sequences
    small_files = sorted([f for f in found_files if "3" in f or "4" in f or "5" in f])
    med_files = sorted([f for f in found_files if "6" in f or "7" in f or "le450_5" in f])
    large_files = sorted([f for f in found_files if f not in small_files and f not in med_files])

    ordered_files = small_files + med_files + large_files
    summary_results = []

    TIMEOUT_LIMIT = 30.0  # Adjustable computational cutoff metric in seconds

    print(f"{'File Name':<20} | {'Vertices':<8} | {'Edges':<8} | {'Target χ':<8} | {'Found χ':<8} | {'Explored':<10} | {'Time (s)':<8} | {'Status'}")
    print("-" * 100)

    for filename in ordered_files:
        file_path = os.path.join(DIMACS_DIR, filename)
        V, E = parse_dimacs_col(file_path)

        if V is None:
            continue

        target_chi = DIMACS_TARGETS[filename]

        # Execute pure downward engine over active graph setup
        status, calc_chi, nodes, elapsed = run_pure_downward_engine(
            V, E, target_chi, timeout_seconds=TIMEOUT_LIMIT
        )

        calc_chi_str = str(calc_chi) if calc_chi is not None else "N/A"
        print(f"{filename:<20} | {len(V):<8} | {len(E)//2:<8} | {target_chi:<8} | {calc_chi_str:<8} | {nodes:<10} | {elapsed:<8.3f} | {status}")

        summary_results.append({
            "name": filename,
            "v_count": len(V),
            "e_count": len(E) // 2,
            "target": target_chi,
            "found": calc_chi_str,
            "states": nodes,
            "time": elapsed,
            "status": status
        })

    # --- FINAL EXPERIMENTAL RUN SUMMARY TABLE ---
    print("\n" + "="*110)
    print(f" {'FINAL DIMACS EXPERIMENTAL RUN SUMMARY TABLE':^108}")
    print("="*110)
    print(f"{'Instance Benchmark File':<25} | {'|V|':<6} | {'|E|':<8} | {'Target χ':<10} | {'Calculated χ':<14} | {'Explored States':<17} | {'Runtime (s)':<12} | {'Outcome Status'}")
    print("-" * 110)
    for res in summary_results:
        print(f"{res['name']:<25} | {res['v_count']:<6} | {res['e_count']:<8} | {res['target']:<10} | {res['found']:<14} | {res['states']:<17} | {res['time']:<12.4f} | {res['status']}")
    print("="*110 + "\n")

if __name__ == "__main__":
    main()

Mounted at /content/drive
Scanning configuration benchmarks folder: /content/drive/MyDrive/GPU/DIMACS_all_ascii

File Name            | Vertices | Edges    | Target χ | Found χ  | Explored   | Time (s) | Status
----------------------------------------------------------------------------------------------------
le450_15a.col        | 450      | 8168     | 15       | 21       | 4463684    | 30.000   | Timeout
le450_25a.col        | 450      | 8260     | 25       | 27       | 3644951    | 30.000   | Timeout
le450_5a.col         | 450      | 5714     | 5        | 13       | 2704942    | 30.000   | Timeout
le450_5b.col         | 450      | 5734     | 5        | 13       | 2755070    | 30.000   | Timeout
myciel3.col          | 11       | 20       | 4        | 4        | 2325       | 0.014    | Success
myciel4.col          | 23       | 71       | 5        | 5        | 929315     | 4.242    | Success
myciel5.col          | 47       | 236      | 6        | 6        | 3241913    | 30.000   | Suc